In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import time


TOPICS = [
    "robotics",
    "ai",
    "sports",
    "semiconductor"
]


def clean_folder_name(name):
    """
    Removes invalid characters from folder names.
    """

    return re.sub(r'[\\/*?:"<>|]', "", name)


def create_topic_folder(topic):
    """
    Creates topic-wise folders inside videos directory.
    """

    folder_path = os.path.join(
        "videos",
        clean_folder_name(topic)
    )

    os.makedirs(folder_path, exist_ok=True)

    return folder_path


def fetch_video_results(topic):
    """
    Collects video-related search results
    from DuckDuckGo.
    """

    search_url = (
        "https://html.duckduckgo.com/html/"
    )

    params = {
        "q": f"{topic} videos"
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        ),
        "Accept-Language": "en-US,en;q=0.9"
    }

    collected_results = []

    try:
        response = requests.post(
            search_url,
            headers=headers,
            data=params,
            timeout=10
        )

        print(f"{topic} -> Status:",
              response.status_code)

        if response.status_code == 200:

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            results = soup.find_all(
                "div",
                class_="result"
            )

            for item in results:

                title_tag = item.find(
                    "a",
                    class_="result__a"
                )

                if title_tag:

                    title = (
                        title_tag.text.strip()
                    )

                    video_url = (
                        title_tag.get("href", "")
                    )

                    if title and video_url:

                        collected_results.append({
                            "topic": topic,
                            "video_title": title,
                            "video_url": video_url
                        })

        else:
            print(f"Search failed for {topic}")

    except Exception as error:
        print(f"Error for {topic}:",
              error)

    return collected_results


all_video_results = []

for topic in TOPICS:

    print(f"\nCollecting videos for: {topic}")

    # creating folders
    create_topic_folder(topic)

    topic_results = fetch_video_results(
        topic
    )

    # merging all collected results
    all_video_results.extend(
        topic_results
    )

    time.sleep(2)


# converting merged results into dataframe
final_df = pd.DataFrame(
    all_video_results
)

# removing duplicate links
final_df = final_df.drop_duplicates(
    subset=["video_url"]
)

# saving outputs
final_df.to_csv(
    "merged_video_results.csv",
    index=False
)

final_df.to_json(
    "merged_video_results.json",
    orient="records",
    indent=4
)

print(
    "\nTotal unique videos collected:",
    len(final_df)
)

print(
    "CSV and JSON files saved successfully"
)


robotics -> Status: 200

ai -> Status: 200

sports -> Status: 200

semiconductor -> Status: 200

Total unique videos collected: 40
CSV and JSON files saved successfully


In [ ]:
+